## **Task 01**

In [1]:
import re
from collections import Counter
import pdfplumber

PDF_PATH = "sample_lab_pdf.pdf"

# ---------- (a) Extract per page ----------
pages = []
with pdfplumber.open(PDF_PATH) as pdf:
    for page in pdf.pages:
        pages.append(page.extract_text() or "")
print(f"Total pages extracted: {len(pages)}")

# ---------- (b) Merge, clean, tokenize ----------
merged = " ".join(pages)
cleaned = re.sub(r"[^\w\s]", "", merged)     # remove punctuation
cleaned = re.sub(r"\d+", "", cleaned)        # remove digits
cleaned = cleaned.lower()
tokens = cleaned.split()
print(f"Total token count: {len(tokens)}")

# ---------- (c) Frequency + stopwords ----------
STOPWORDS = {
    "the","a","an","and","or","but","if","then","else","of","to","in","on",
    "at","by","for","with","about","as","is","are","was","were","be","been",
    "being","this","that","these","those","it","its","from","not","no","so",
    "can","may","more","most","such","than","then","them","they","we","you",
    "our","your","their","there","here","which","who","whom","what","when"
}

freq = Counter(t for t in tokens if t not in STOPWORDS)
print("\nTop 10 most frequent words:")
for word, count in freq.most_common(10):
    print(f"  {word:<15} {count}")

# ---------- (d) Keyword sentence search ----------
keyword = input("\nEnter keyword to search: ").lower()
found_any = False
for page_no, text in enumerate(pages, start=1):
    sentences = re.split(r"(?<=[.!?])\s+", text)
    for s in sentences:
        if keyword in s.lower():
            print(f"[Page {page_no}] {s.strip()}")
            found_any = True
if not found_any:
    print(f"No sentences containing '{keyword}' were found.")

Total pages extracted: 6
Total token count: 642

Top 10 most frequent words:
  text            13
  python          7
  data            6
  analysis        6
  language        5
  processing      5
  nlp             5
  document        5
  words           5
  into            4
No sentences containing '12' were found.


## **Task 02**

In [3]:
import re, csv
import pdfplumber

PDF_PATH = "sample_lab_pdf.pdf"

class PhraseNotFoundError(Exception):
    pass

# ---------- (a) Build pages_data ----------
pages_data = []
with pdfplumber.open(PDF_PATH) as pdf:
    for i, page in enumerate(pdf.pages, start=1):
        raw = page.extract_text() or ""
        pages_data.append({
            "page_number": i,
            "raw_text": raw,
            "word_count": len(raw.split()),
            "char_count": len(re.sub(r"\s", "", raw)),
        })

print("First 2 entries:")
for entry in pages_data[:2]:
    preview = {k: (v[:60] + "..." if k == "raw_text" and len(v) > 60 else v)
               for k, v in entry.items()}
    print(preview)

# ---------- (b) Highest / lowest / average ----------
highest = max(pages_data, key=lambda p: p["word_count"])
print(f"\nHighest word count page: {highest['page_number']} "
      f"({highest['word_count']} words)")
print("Full content:\n", highest["raw_text"])

nonzero = [p for p in pages_data if p["word_count"] > 0]
lowest = min(nonzero, key=lambda p: p["word_count"])
print(f"\nLowest non-zero page: {lowest['page_number']} "
      f"({lowest['word_count']} words)")

avg = round(sum(p["word_count"] for p in pages_data) / len(pages_data), 2)
print(f"Average word count: {avg}")

# ---------- (c) Case-insensitive phrase search ----------
phrase = input("\nEnter search phrase: ").lower()
matches = []
for p in pages_data:
    cnt = p["raw_text"].lower().count(phrase)
    if cnt:
        matches.append((p["page_number"], cnt))

try:
    if not matches:
        raise PhraseNotFoundError(f"Phrase '{phrase}' not found in PDF.")
    for page_no, cnt in matches:
        print(f"Page {page_no}: {cnt} occurrence(s)")
except PhraseNotFoundError as e:
    print("Error:", e)

# ---------- (d) CSV write + re-read ----------
with open("pdf_summary.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["page_number", "word_count", "char_count"])
    w.writeheader()
    for p in pages_data:
        w.writerow({k: p[k] for k in ("page_number", "word_count", "char_count")})

print("\nRe-reading pdf_summary.csv:")
with open("pdf_summary.csv", encoding="utf-8") as f:
    for row in csv.reader(f):
        print(row)

First 2 entries:
{'page_number': 1, 'raw_text': 'Introduction to Natural Language Processing\nNatural Language...', 'word_count': 124, 'char_count': 706}
{'page_number': 2, 'raw_text': 'Document Processing with Python\nDocument processing means ex...', 'word_count': 130, 'char_count': 671}

Highest word count page: 2 (130 words)
Full content:
 Document Processing with Python
Document processing means extracting useful information from files such as PDF and DOCX
documents. Python libraries like PyPDF2 and python-docx make it easy to read text,
paragraphs, and tables from these files. The extracted data can then be stored in lists,
dictionaries, or pandas DataFrames.
A typical workflow has four steps: open the file, extract the text, clean the text, and analyze the
result. Each step can be tested separately, which makes the whole process reliable and
reproducible. Data quality matters more than model complexity in most projects.
For example, a report with 25 pages can be converted into 2

## **Task 03**

In [4]:
import re
import pdfplumber
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords", quiet=True)
PDF_PATH = "sample_lab_pdf.pdf"

with pdfplumber.open(PDF_PATH) as pdf:
    raw = " ".join(p.extract_text() or "" for p in pdf.pages)

# ---------- (a) Cleaning ----------
before = len(raw)
step1 = re.sub(r"http\S+|www\.\S+", "", raw)
step2 = re.sub(r"\d+", "", step1)
step3 = re.sub(r"[^a-zA-Z\s.]", "", step2)
step4 = re.sub(r"\s+", " ", step3).strip()
print(f"Character count before: {before}")
print(f"Character count after : {len(step4)}")

# ---------- (b) Sentence splitting ----------
lowered = step4.lower()
sentences = [s.strip() for s in re.split(r"[.!?]", lowered)]
before_empty = len(sentences)
sentences = [s for s in sentences if s]
print(f"\nSentences before removing empty: {before_empty}")
print(f"Sentences after removing empty : {len(sentences)}")

# ---------- (c) Stopword filter (≥5 tokens) ----------
sw = set(stopwords.words("english"))
filtered = []
for s in sentences:
    toks = [t for t in s.split() if t not in sw]
    if len(toks) >= 5:
        filtered.append(s)

print(f"\nSentences before stopword filter: {len(sentences)}")
print(f"Sentences after  stopword filter: {len(filtered)}")
print("\n3 sample sentences:")
for s in filtered[:3]:
    print(" •", s)

# ---------- (d) Write + re-read ----------
with open("nlp_ready_output.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(filtered))

with open("nlp_ready_output.txt", encoding="utf-8") as f:
    lines = [ln.rstrip("\n") for ln in f if ln.strip()]

print(f"\nTotal lines written: {len(lines)}")
if lines:
    print("Longest  :", max(lines, key=len))
    print("Shortest :", min(lines, key=len))

Character count before: 4117
Character count after : 3959

Sentences before removing empty: 43
Sentences after removing empty : 42

Sentences before stopword filter: 42
Sentences after  stopword filter: 41

3 sample sentences:
 • introduction to natural language processing natural language processing nlp is a field of artificial intelligence that helps machines understand human language
 • it combines linguistics computer science and statistics to process text and speech
 • python is the most popular language for nlp because of libraries such as nltk spacy and gensim

Total lines written: 41
Longest  : why is text so difficult for computers human language is ambiguous full of idioms and constantly changing a good nlp pipeline must therefore clean tokenize and normalize text before any model can learn from it
Shortest : sentence splitting is another common task


## **Task 04**

In [7]:
import pprint
from docx import Document

DOCX_PATH = "sample_lab_document.docx"
doc = Document(DOCX_PATH)

# ---------- (a) ----------
paragraphs = [{"text": p.text, "style": p.style.name} for p in doc.paragraphs]
print("Total paragraphs:", len(paragraphs))
print("Unique styles:", sorted({p["style"] for p in paragraphs}))

# ---------- (b) ----------
headings, body, other = [], [], []
for p in paragraphs:
    s = p["style"]
    if s.startswith("Heading"):
        headings.append(p)
    elif s == "Normal" or "Body" in s:
        body.append(p)
    else:
        other.append(p)
print(f"\nHeadings: {len(headings)}  Body: {len(body)}  Other/Empty: {len(other)}")

# ---------- (c) ----------
qualifying = []
for p in body:
    words = p["text"].split()
    if len(words) > 5:
        qualifying.append({
            "text": p["text"],
            "word_count": len(words),
            "char_count": len(p["text"].replace(" ", "")),
        })

if qualifying:
    hi = max(qualifying, key=lambda x: x["word_count"])
    lo = min(qualifying, key=lambda x: x["word_count"])
    mean = round(sum(q["word_count"] for q in qualifying) / len(qualifying), 2)
    print("\nHighest word count paragraph:")
    print(" ", hi["text"])
    print("\nLowest (>5 words) paragraph:")
    print(" ", lo["text"])
    print(f"\nMean word count: {mean}")

# ---------- (d) ----------
style_map = {}
for p in paragraphs:
    style_map.setdefault(p["style"], {"count": 0, "texts": []})
    style_map[p["style"]]["count"] += 1
    style_map[p["style"]]["texts"].append(p["text"])

sorted_map = dict(sorted(style_map.items(),
                         key=lambda kv: kv[1]["count"], reverse=True))
trimmed = {
    s: {"count": d["count"], "texts": d["texts"][:2] if d["count"] > 2 else d["texts"]}
    for s, d in sorted_map.items()
}
print("\nStyle summary:")
pprint.pprint(trimmed, width=100)

Total paragraphs: 15
Unique styles: ['Heading 1', 'Heading 2', 'List Bullet', 'Normal', 'Title']

Headings: 4  Body: 8  Other/Empty: 3

Highest word count paragraph:
  The objective of this lab is to practise reading structured documents and converting tables into pandas DataFrames for further analysis.

Lowest (>5 words) paragraph:
  Text processing helps identify words, paragraphs, and repeated information in large documents.

Mean word count: 14.0

Style summary:
{'Heading 1': {'count': 3, 'texts': ['Introduction', 'Student Results']},
 'Heading 2': {'count': 1, 'texts': ['Objectives']},
 'List Bullet': {'count': 2,
                 'texts': ['Extract paragraphs and their styles',
                           'Analyze paragraph length statistics']},
 'Normal': {'count': 8,
            'texts': ['This report demonstrates how Python can extract text, paragraphs, and '
                      'tables from Word documents.',
                      'Text processing helps identify words, paragr

## **Task 05**

In [8]:
import pandas as pd
from docx import Document
from docx.text.paragraph import Paragraph

DOCX_PATH = "sample_lab_document.docx"
doc = Document(DOCX_PATH)

# ---------- (a) ----------
tables = []
for t in doc.tables:
    data = [[c.text.strip() for c in row.cells] for row in t.rows]
    tables.append(data)

print("Total tables found:", len(tables))
for i, t in enumerate(tables, 1):
    rows, cols = len(t), len(t[0]) if t else 0
    print(f"  Table {i}: {rows} x {cols}")

# ---------- (b) ----------
if tables:
    header, rows = tables[0][0], tables[0][1:]
    df = pd.DataFrame(rows, columns=header)
    print("\nDataFrame:")
    print(df)

    print("\nEmpty/None cells per column:")
    for col in df.columns:
        empty = (df[col].isna() | (df[col] == "")).sum()
        print(f"  {col}: {empty}")

# ---------- (c) ----------
pre_table = []
for child in doc.element.body.iterchildren():
    if child.tag.endswith("}tbl"):
        break
    if child.tag.endswith("}p"):
        pre_table.append(Paragraph(child, doc).text)
print("\nPre-table paragraphs count:", len(pre_table))

# ---------- (d) ----------
with open("docx_extracted.txt", "w", encoding="utf-8") as f:
    f.write("--- PRE-TABLE PARAGRAPHS ---\n")
    f.write("\n".join(pre_table) + "\n\n")
    f.write("--- TABLE DATA ---\n")
    if tables:
        for row in tables[0]:
            f.write(" | ".join(row) + "\n")

print("Wrote docx_extracted.txt")

Total tables found: 2
  Table 1: 4 x 3
  Table 2: 3 x 2

DataFrame:
    Name Score Status
0    Ali    85   Pass
1   Sara         Pass
2  Usman    42       

Empty/None cells per column:
  Name: 0
  Score: 1
  Status: 1

Pre-table paragraphs count: 12
Wrote docx_extracted.txt


## **Task 06**

In [9]:
import re
from collections import Counter
from docx import Document

DOC_A = "sample_document_A.docx"
DOC_B = "sample_document_B.docx"

def load(path):
    return [{"text": p.text, "style": p.style.name}
            for p in Document(path).paragraphs]

A, B = load(DOC_A), load(DOC_B)

# ---------- (a) Heading comparison ----------
head_a = {p["text"] for p in A if p["style"].startswith("Heading")}
head_b = {p["text"] for p in B if p["style"].startswith("Heading")}
print("Common headings :", head_a & head_b)
print("Only in A        :", head_a - head_b)
print("Only in B        :", head_b - head_a)

# ---------- (b) Dedup Normal paragraphs ----------
normal = [p["text"] for p in A + B if p["style"] == "Normal"]
print(f"\nBefore dedup: {len(normal)}")

seen, deduped = set(), []
for t in normal:
    key = t.strip().lower()
    if key and key not in seen:
        seen.add(key)
        deduped.append(t)
print(f"After dedup : {len(deduped)}")

# ---------- (c) Word frequency ----------
STOP = {
    "the","a","an","and","or","but","if","then","of","to","in","on","at",
    "by","for","with","is","are","was","were","be","been","this","that",
    "it","its","from","not","no","so","as","used","test"
}
freq = Counter()
for para in deduped:
    clean = re.sub(r"[^\w\s]", "", para.lower())
    freq.update(w for w in clean.split() if w not in STOP)

print("\nTop 20 most frequent words:")
for w, c in freq.most_common(20):
    print(f"  {w:<15} {c}")

# ---------- (d) Summary ----------
total_headings = len(head_a | head_b)
unique_body = len(deduped)
vocab = len(freq)

lengths = [len(p.split()) for p in deduped]
avg_len = round(sum(lengths) / len(lengths), 2) if lengths else 0
longest = max(deduped, key=lambda p: len(p.split())) if deduped else ""

print("\n=== SUMMARY ===")
print("Total headings (union)      :", total_headings)
print("Unique body paragraphs      :", unique_body)
print("Unique word types           :", vocab)
print("Avg paragraph length (words):", avg_len)
print("Longest paragraph           :", longest)

Common headings : {'Shared Topic', 'Conclusion', 'Introduction'}
Only in A        : {'Methodology'}
Only in B        : {'Discussion'}

Before dedup: 12
After dedup : 8

Top 20 most frequent words:
  document        4
  documents       3
  analysis        2
  text            2
  python          1
  widely          1
  mining          1
  data            1
  preparation     1
  tasks           1
  paragraph       1
  appears         1
  both            1
  deduplication   1
  natural         1
  language        1
  processing      1
  turns           1
  raw             1
  into            1

=== SUMMARY ===
Total headings (union)      : 5
Unique body paragraphs      : 8
Unique word types           : 58
Avg paragraph length (words): 11.38
Longest paragraph           : Python is widely used for document analysis, text mining, and data preparation tasks.


## **Task 07**

In [10]:
import requests

USERS = "https://jsonplaceholder.typicode.com/users"
POSTS = "https://jsonplaceholder.typicode.com/posts"

# ---------- (a) ----------
r_users, r_posts = requests.get(USERS), requests.get(POSTS)
if r_users.status_code != 200:
    raise ConnectionError(f"Users returned {r_users.status_code}")
if r_posts.status_code != 200:
    raise ConnectionError(f"Posts returned {r_posts.status_code}")
users, posts = r_users.json(), r_posts.json()

# ---------- (b) ----------
joined = {
    u["name"]: {
        "email": u["email"],
        "company": u["company"]["name"],
        "company_bs": u["company"]["bs"],
        "city": u["address"]["city"],
        "post_count": 0,
        "titles": [],
    } for u in users
}
uid_to_name = {u["id"]: u["name"] for u in users}

for p in posts:
    name = uid_to_name.get(p["userId"])
    if name:
        joined[name]["post_count"] += 1
        joined[name]["titles"].append(p["title"])

print("First 3 users:")
for name in list(joined)[:3]:
    info = joined[name]
    print(f"  {name} ({info['email']}) — {info['post_count']} posts")

# ---------- (c) ----------
top = max(joined.items(), key=lambda kv: kv[1]["post_count"])
name, info = top
print(f"\nTop poster: {name}")
print("Email   :", info["email"])
print("Company :", info["company"])
print("Titles:")
for i, t in enumerate(info["titles"], 1):
    print(f"  {i}. {t}")

# ---------- (d) ----------
print("\nFiltered users:")
for name, info in joined.items():
    bs_match = "synergies" in info["company_bs"].lower()
    city_match = len(info["city"]) > 8
    if bs_match or city_match:
        print(f"  {name:<25} | {info['city']:<15} | posts={info['post_count']}")

First 3 users:
  Leanne Graham (Sincere@april.biz) — 10 posts
  Ervin Howell (Shanna@melissa.tv) — 10 posts
  Clementine Bauch (Nathan@yesenia.net) — 10 posts

Top poster: Leanne Graham
Email   : Sincere@april.biz
Company : Romaguera-Crona
Titles:
  1. sunt aut facere repellat provident occaecati excepturi optio reprehenderit
  2. qui est esse
  3. ea molestias quasi exercitationem repellat qui ipsa sit aut
  4. eum et est occaecati
  5. nesciunt quas odio
  6. dolorem eum magni eos aperiam quia
  7. magnam facilis autem
  8. dolorem dolore est ipsam
  9. nesciunt iure omnis dolorem tempora et accusantium
  10. optio molestias id quia eum

Filtered users:
  Leanne Graham             | Gwenborough     | posts=10
  Ervin Howell              | Wisokyburgh     | posts=10
  Clementine Bauch          | McKenziehaven   | posts=10
  Patricia Lebsack          | South Elvis     | posts=10
  Chelsey Dietrich          | Roscoeview      | posts=10
  Mrs. Dennis Schulist      | South Christy   | pos

## **Task 08**

In [11]:
import requests

# ---------- (a) ----------
url = "https://jsonplaceholder.typicode.com/todos"
r = requests.get(url)

if r.status_code != 200:
    print(f"HTTP error: {r.status_code}")
    raise SystemExit
todos = r.json()
if not isinstance(todos, list):
    print("Response is not a list.")
    raise SystemExit
if not todos:
    print("No records returned.")
    raise SystemExit

print(f"Total todo records: {len(todos)}")

# ---------- (b) ----------
stats = {}
for t in todos:
    uid = t["userId"]
    stats.setdefault(uid, {"total": 0, "completed": 0, "incomplete": 0})
    stats[uid]["total"] += 1
    stats[uid]["completed" if t["completed"] else "incomplete"] += 1

for uid, s in sorted(stats.items()):
    ok = s["completed"] + s["incomplete"] == s["total"]
    print(f"User {uid}: {s}  -> consistency: {ok}")

# ---------- (c) ----------
rates = {u: round(s["completed"] / s["total"] * 100, 1)
         for u, s in stats.items()}
best  = max(rates.items(), key=lambda kv: kv[1])
worst = min(rates.items(), key=lambda kv: kv[1])
avg   = round(sum(rates.values()) / len(rates), 1)

print(f"\nHighest completion: user {best[0]}  -> {best[1]}%")
print(f"Lowest  completion: user {worst[0]} -> {worst[1]}%")
print(f"Overall average   : {avg}%")

# ---------- (d) ----------
name_lookup = {u["id"]: u["name"]
               for u in requests.get("https://jsonplaceholder.typicode.com/users").json()}

rows = [(uid, name_lookup.get(uid, "?"), s["total"], s["completed"], rates[uid])
        for uid, s in stats.items()]
rows.sort(key=lambda r: r[4], reverse=True)

print("\n" + "Rank".ljust(6) + "Name".ljust(25) +
      "Total".ljust(8) + "Completed".ljust(11) + "Rate %")
print("-" * 62)
for i, (uid, name, total, comp, rate) in enumerate(rows, 1):
    print(str(i).ljust(6) + name.ljust(25) +
          str(total).ljust(8) + str(comp).ljust(11) + f"{rate}")

Total todo records: 200
User 1: {'total': 20, 'completed': 11, 'incomplete': 9}  -> consistency: True
User 2: {'total': 20, 'completed': 8, 'incomplete': 12}  -> consistency: True
User 3: {'total': 20, 'completed': 7, 'incomplete': 13}  -> consistency: True
User 4: {'total': 20, 'completed': 6, 'incomplete': 14}  -> consistency: True
User 5: {'total': 20, 'completed': 12, 'incomplete': 8}  -> consistency: True
User 6: {'total': 20, 'completed': 6, 'incomplete': 14}  -> consistency: True
User 7: {'total': 20, 'completed': 9, 'incomplete': 11}  -> consistency: True
User 8: {'total': 20, 'completed': 11, 'incomplete': 9}  -> consistency: True
User 9: {'total': 20, 'completed': 8, 'incomplete': 12}  -> consistency: True
User 10: {'total': 20, 'completed': 12, 'incomplete': 8}  -> consistency: True

Highest completion: user 5  -> 60.0%
Lowest  completion: user 4 -> 30.0%
Overall average   : 45.0%

Rank  Name                     Total   Completed  Rate %
-------------------------------------

## **Task 09**

In [12]:
import math
import pandas as pd
import requests

# ---------- (a) ----------
def flatten_user(u: dict) -> dict:
    return {
        "id": u["id"],
        "name": u["name"],
        "username": u["username"],
        "email": u["email"],
        "city": u["address"]["city"],
        "zipcode": u["address"]["zipcode"],
        "lat": u["address"]["geo"]["lat"],
        "lng": u["address"]["geo"]["lng"],
        "company_name": u["company"]["name"],
        "company_catchphrase": u["company"]["catchPhrase"],
    }

users = requests.get("https://jsonplaceholder.typicode.com/users").json()
flat = [flatten_user(u) for u in users]
print("Flattened users:")
for u in flat:
    print(" ", u)

# ---------- (b) ----------
df = pd.DataFrame(flat)
print("\nDataFrame:")
print(df)

df["lat"] = df["lat"].astype(float)
df["lng"] = df["lng"].astype(float)
df["distance_from_origin"] = df.apply(
    lambda r: math.sqrt(r["lat"] ** 2 + r["lng"] ** 2), axis=1)

farthest = df.loc[df["distance_from_origin"].idxmax()]
print(f"\nFarthest from origin: {farthest['name']} "
      f"({farthest['distance_from_origin']:.4f})")

# ---------- (c) ----------
north = df[df["lat"] > 0]
south = df[df["lat"] < 0]
print("\nNorthern hemisphere:", list(north["name"]))
print("Southern hemisphere:", list(south["name"]))

north_avg = north["distance_from_origin"].mean() if len(north) else 0
south_avg = south["distance_from_origin"].mean() if len(south) else 0
print(f"\nNorth avg distance: {north_avg:.4f}")
print(f"South avg distance: {south_avg:.4f}")
print("Hemisphere with higher avg distance:",
      "North" if north_avg > south_avg else "South")

# ---------- (d) ----------
posts    = requests.get("https://jsonplaceholder.typicode.com/posts").json()
comments = requests.get("https://jsonplaceholder.typicode.com/comments").json()

cc = {}
for c in comments:
    cc[c["postId"]] = cc.get(c["postId"], 0) + 1

result = sorted(
    [(p["id"], p["title"][:40], cc.get(p["id"], 0)) for p in posts],
    key=lambda x: x[2], reverse=True
)

print("\nTop 5 most-commented posts:")
for row in result[:5]:
    print(" ", row)
print("\nBottom 5 least-commented posts:")
for row in result[-5:]:
    print(" ", row)

Flattened users:
  {'id': 1, 'name': 'Leanne Graham', 'username': 'Bret', 'email': 'Sincere@april.biz', 'city': 'Gwenborough', 'zipcode': '92998-3874', 'lat': '-37.3159', 'lng': '81.1496', 'company_name': 'Romaguera-Crona', 'company_catchphrase': 'Multi-layered client-server neural-net'}
  {'id': 2, 'name': 'Ervin Howell', 'username': 'Antonette', 'email': 'Shanna@melissa.tv', 'city': 'Wisokyburgh', 'zipcode': '90566-7771', 'lat': '-43.9509', 'lng': '-34.4618', 'company_name': 'Deckow-Crist', 'company_catchphrase': 'Proactive didactic contingency'}
  {'id': 3, 'name': 'Clementine Bauch', 'username': 'Samantha', 'email': 'Nathan@yesenia.net', 'city': 'McKenziehaven', 'zipcode': '59590-4157', 'lat': '-68.6102', 'lng': '-47.0653', 'company_name': 'Romaguera-Jacobson', 'company_catchphrase': 'Face to face bifurcated interface'}
  {'id': 4, 'name': 'Patricia Lebsack', 'username': 'Karianne', 'email': 'Julianne.OConner@kory.org', 'city': 'South Elvis', 'zipcode': '53919-4257', 'lat': '29.457